
# 練習問題: マンデルブロ集合と schedule の選択

## 目標

仕事量が反復ごとに大きく異なるループを並列化し, `schedule` 節がなぜ重要かを体感する. 静的割り当て (`schedule(static)`) では負荷の不均衡で遅くなり, 動的割り当て (`schedule(dynamic)`) では負荷が均されて速くなることを `time` で確認する.

## 背景: マンデルブロ集合

複素数 `c` ごとに, `z = 0` から始めて `z ← z^2 + c` を繰り返す. `|z|^2 > 4` になったら「脱出」とみなし, そのときの反復回数を記録する. 集合内部の点は最後 (`maxiter`) まで脱出しないため反復回数が多く, 外側の点はすぐ脱出する. つまり**画素ごとに仕事量が大きく異なる**.

このプログラムは `W × H` の格子 (既定 1000×1000), 最大反復 `maxiter` (既定 1000) で各画素の脱出反復数を配列 `cnt` に格納し, 並列ループの後で総反復数を逐次に集計する (足し込みの競合を避けるため).

## 課題

`mandelbrot.cpp` (または `mandelbrot.f90`) の画素ループ (`px` ループ) を並列化せよ.

コメント `TODO` の指示に従って **OpenMP の指示を追加** する.

- C++: `px` ループの直前に `#pragma omp parallel for schedule(dynamic)` を1行加える.
- Fortran: `px` ループを `!$omp parallel do schedule(dynamic) private(...)` と `!$omp end parallel do` で囲む (`private` 節は雛形のコメントの通り).

画素ごとの仕事量が大きく異なるため, 各スレッドに均等な数の反復を割り当てる `schedule(static)` では, 集合内部を担当したスレッドだけ重くなり全体が遅くなる. 一方 `schedule(dynamic)` は終わったスレッドが次の仕事を取りに行くので負荷が均される.

総反復数 (`total iterations`) は **スケジュールによらず同じ値**になる (正しさの確認に使える).

## コンパイルと実行

```
# C++
nvc++ -fast -mp=multicore mandelbrot.cpp -o mandelbrot.exe

# Fortran
nvfortran -fast -mp=multicore mandelbrot.f90 -o mandelbrot.exe
```

```
OMP_NUM_THREADS=8 ./mandelbrot.exe
```

`schedule(dynamic)` で動かしたら, 指示を `schedule(static)` に書き換えて再コンパイルし, `time` で実行時間を比べよ:

```
OMP_NUM_THREADS=8 time ./mandelbrot.exe
```

## 期待される結果

```
W=1000 H=1000 maxiter=1000
total iterations = ...   (スケジュールによらず一定)
sample cnt: top-left=... center=1000 bottom-right=...
```

- `total iterations` は `schedule(dynamic)` でも `schedule(static)` でも **同じ**になる (正しさの確認).
- 中心 (center) の画素は集合内部なので反復数が `maxiter` (=1000) に達する.
- 実行時間は `schedule(dynamic)` の方が `schedule(static)` より短くなる (負荷分散の効果). スレッド数を増やすほど差が顕著になる.



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ mandelbrot.cpp
#include <cstdio>
#include <cstdlib>

int main(int argc, char **argv) {
  // 画像サイズと最大反復数
  int W = (argc > 1) ? atoi(argv[1]) : 1000;
  int H = (argc > 2) ? atoi(argv[2]) : 1000;
  int maxiter = (argc > 3) ? atoi(argv[3]) : 1000;

  int *cnt = (int *)malloc((size_t)W * H * sizeof(int));

  // 複素平面の描画範囲
  const double xmin = -2.0, xmax = 1.0;
  const double ymin = -1.5, ymax = 1.5;

  // 各ピクセルの脱出反復数を計算する.
  // 内部の点は maxiter まで回るため画素ごとの仕事量が大きく異なる (負荷が不均一).
  // TODO: 下の行 (px ループ) の直前に #pragma omp parallel for schedule(dynamic) を追加せよ. 仕事量が画素ごとに大きく異なるため, dynamic スケジュールが負荷を均す.
  for (int px = 0; px < W * H; px++) {
    int i = px / W;  // 行
    int j = px % W;  // 列
    double cx = xmin + (xmax - xmin) * j / (W - 1);
    double cy = ymin + (ymax - ymin) * i / (H - 1);
    // z = z^2 + c を |z|^2 > 4 か maxiter まで反復 (複素数を手で展開)
    double zr = 0.0, zi = 0.0;
    int it = 0;
    while (it < maxiter && zr * zr + zi * zi <= 4.0) {
      double zr2 = zr * zr - zi * zi + cx;
      double zi2 = 2.0 * zr * zi + cy;
      zr = zr2;
      zi = zi2;
      it++;
    }
    cnt[px] = it;
  }

  // 並列ループの後で総反復数を逐次に集計する (共有変数への足し込みによる競合を避ける)
  long long total = 0;
  for (int px = 0; px < W * H; px++) {
    total += cnt[px];
  }

  printf("W=%d H=%d maxiter=%d\n", W, H, maxiter);
  printf("total iterations = %lld\n", total);
  printf("sample cnt: top-left=%d center=%d bottom-right=%d\n",
         cnt[0], cnt[(H / 2) * W + W / 2], cnt[W * H - 1]);

  free(cnt);
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore mandelbrot.cpp -o mandelbrot_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./mandelbrot_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ mandelbrot.f90
program mandelbrot
  implicit none
  integer :: W, H, maxiter
  integer :: px, i, j, it
  integer, allocatable :: cnt(:)
  real(8) :: xmin, xmax, ymin, ymax, cx, cy, zr, zi, zr2, zi2
  integer(8) :: total
  character(len=32) :: arg

  ! 画像サイズと最大反復数
  W = 1000; H = 1000; maxiter = 1000
  if (command_argument_count() >= 1) then
     call get_command_argument(1, arg); read(arg, *) W
  end if
  if (command_argument_count() >= 2) then
     call get_command_argument(2, arg); read(arg, *) H
  end if
  if (command_argument_count() >= 3) then
     call get_command_argument(3, arg); read(arg, *) maxiter
  end if

  allocate(cnt(0:W*H-1))

  ! 複素平面の描画範囲
  xmin = -2.0d0; xmax = 1.0d0
  ymin = -1.5d0; ymax = 1.5d0

  ! 各ピクセルの脱出反復数を計算する.
  ! 内部の点は maxiter まで回るため画素ごとの仕事量が大きく異なる (負荷が不均一).
  ! TODO: 下の px ループを !$omp parallel do schedule(dynamic) private(i, j, cx, cy, zr, zi, zr2, zi2, it) ... !$omp end parallel do で囲め. 仕事量が画素ごとに大きく異なるため, dynamic スケジュールが負荷を均す.
  do px = 0, W*H - 1
     i = px / W   ! 行
     j = mod(px, W)  ! 列
     cx = xmin + (xmax - xmin) * j / (W - 1)
     cy = ymin + (ymax - ymin) * i / (H - 1)
     ! z = z^2 + c を |z|^2 > 4 か maxiter まで反復 (複素数を手で展開)
     zr = 0.0d0; zi = 0.0d0
     it = 0
     do while (it < maxiter .and. zr*zr + zi*zi <= 4.0d0)
        zr2 = zr*zr - zi*zi + cx
        zi2 = 2.0d0 * zr * zi + cy
        zr = zr2
        zi = zi2
        it = it + 1
     end do
     cnt(px) = it
  end do
  ! TODO: 上で始めた parallel do を閉じる (!$omp end parallel do).

  ! 並列ループの後で総反復数を逐次に集計する (共有変数への足し込みによる競合を避ける)
  total = 0
  do px = 0, W*H - 1
     total = total + cnt(px)
  end do

  print "(a,i0,a,i0,a,i0)", "W=", W, " H=", H, " maxiter=", maxiter
  print "(a,i0)", "total iterations = ", total
  print "(a,i0,a,i0,a,i0)", "sample cnt: top-left=", cnt(0), &
       " center=", cnt((H/2)*W + W/2), " bottom-right=", cnt(W*H - 1)

  deallocate(cnt)
end program mandelbrot

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore mandelbrot.f90 -o mandelbrot_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./mandelbrot_f90.exe


# 4. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:mandelbrot.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 4-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 4-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 4-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:mandelbrot.cpp}

コマンドと実行結果:
{bash[-1]}

## 4-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:mandelbrot.cpp}